In [1]:
import urllib.request; exec(urllib.request.urlopen('https://aic-data.aiffel.io/api/colab/setup?t=3vvc47bz').read().decode())

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit



⏳  터널 준비 확인 중...

✅  터널 생성 완료!
🔗  URL: https://numerical-contributor-managers-activities.trycloudflare.com

아래 [URL 복사] 버튼을 누른 뒤 웹앱 연결창에 붙여넣으세요. (이 탭은 열어두세요)


✅ 웹앱에 자동 연결 요청을 보냈습니다. 잠시 후 웹앱 화면이 연결됩니다.


In [4]:
# ─────────────────────────────────────────────
# 환경 준비 — 라이브러리 + 한글 폰트 + 시드 고정
# ─────────────────────────────────────────────
# 필요 시 아래 주석을 해제해 설치하세요.
# !pip install numpy pandas matplotlib seaborn missingno -q

import platform
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno   # 결측 패턴 시각화 전용 라이브러리 (Part 3에서 사용)

warnings.filterwarnings("ignore")

# 재현성: 같은 난수를 항상 같게 만들어 결과가 매번 동일하도록 고정합니다.
np.random.seed(42)

# 한글 폰트 설정 (운영체제별 분기)
system = platform.system()
if system == "Darwin":
    plt.rcParams["font.family"] = "AppleGothic"
elif system == "Windows":
    plt.rcParams["font.family"] = "Malgun Gothic"
else:
    plt.rcParams["font.family"] = "DejaVu Sans"

plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.figsize"] = (10, 5)
sns.set_style("whitegrid")

print("준비 완료! 라이브러리 버전을 확인합니다.")
print("numpy :", np.__version__)
print("pandas:", pd.__version__)

준비 완료! 라이브러리 버전을 확인합니다.
numpy : 2.0.2
pandas: 2.2.2


In [5]:
# ─────────────────────────────────────────────
# 모두마켓 데이터 생성 (D+003 버전)
# - 어제 발견한 오염을 그대로 가져오되, 결측/이상치를 더 다양하게 심습니다.
# - 이 셀 하나로 오늘 쓸 데이터가 전부 준비됩니다 (이 노트북은 단독 실행 가능).
# ─────────────────────────────────────────────
np.random.seed(42)

# 1) 고객(customers)
n_customers = 300
customers = pd.DataFrame({
    "customer_id": [f"C{str(i).zfill(4)}" for i in range(1, n_customers + 1)],
    "age": np.random.normal(35, 9, n_customers).round().astype(int),
    "gender": np.random.choice(["M", "F"], n_customers),
    "region": np.random.choice(["서울", "경기", "부산", "인천", "대구"], n_customers),
    "membership": np.random.choice(["basic", "premium", "vip"], n_customers, p=[0.6, 0.3, 0.1]),
    "income": np.random.choice([2500, 3500, 4500, 6000, 8500], n_customers).astype(float),
})

# 오염 심기 — 나이 이상치
customers.loc[5, "age"] = 999            # 입력 실수로 보이는 이상치
customers.loc[10, "age"] = -3            # 음수 나이(불가능한 값)
customers.loc[15, "age"] = 120           # 매우 큰 값 (이상치 후보)

# 오염 심기 — 성별 결측 (관찰자가 응답을 안 받았을 가능성: MCAR에 가까움)
customers.loc[[20, 21, 22, 70, 120], "gender"] = np.nan

# 오염 심기 — 소득 결측 (고소득 고객일수록 응답 거절 → MNAR 신호)
high_income_mask = customers["income"] >= 6000
high_idx = customers[high_income_mask].sample(frac=0.4, random_state=1).index
customers.loc[high_idx, "income"] = np.nan

# 2) 상품(products)
categories = ["패션", "뷰티", "식품", "가전", "도서"]
n_products = 40
products = pd.DataFrame({
    "product_id": [f"P{str(i).zfill(3)}" for i in range(1, n_products + 1)],
    "category": np.random.choice(categories, n_products),
    "price": np.random.choice([9900, 19900, 29900, 49900, 89900, 129900], n_products),
})

# 3) 주문(orders)
n_orders = 2000
order_customer = np.random.choice(customers["customer_id"], n_orders)
order_product = np.random.choice(products["product_id"], n_orders)
price_map = products.set_index("product_id")["price"]
quantity = np.random.choice([1, 1, 1, 2, 2, 3], n_orders)
amount = price_map.loc[order_product].values * quantity

orders = pd.DataFrame({
    "order_id": [f"O{str(i).zfill(5)}" for i in range(1, n_orders + 1)],
    "customer_id": order_customer,
    "product_id": order_product,
    "category": products.set_index("product_id").loc[order_product, "category"].values,
    "quantity": quantity,
    "amount": amount.astype(float),
    "channel": np.random.choice(["web", "app"], n_orders, p=[0.5, 0.5]),
})

# 오염 심기 — 금액 결측 (일부 채널에서만 결측이 더 잦음 → MAR 신호)
app_mask = orders["channel"] == "app"
app_idx = orders[app_mask].sample(frac=0.04, random_state=2).index
web_idx = orders[~app_mask].sample(frac=0.01, random_state=3).index
orders.loc[app_idx.union(web_idx), "amount"] = np.nan

# 오염 심기 — 수량 이상치
orders.loc[7, "quantity"] = 100         # 비정상적으로 큰 주문 수량
orders.loc[1900, "quantity"] = 50       # 또 하나의 극단값
orders.loc[1500, "amount"] = 5_000_000  # 금액 자체의 극단값

print("모두마켓 데이터 생성 완료")
print("customers:", customers.shape, "| products:", products.shape, "| orders:", orders.shape)
print("\n[빠르게 둘러보기]")
print("customers 결측:", customers.isnull().sum().sum(), "건")
print("orders 결측:", orders.isnull().sum().sum(), "건")

모두마켓 데이터 생성 완료
customers: (300, 6) | products: (40, 3) | orders: (2000, 7)

[빠르게 둘러보기]
customers 결측: 51 건
orders 결측: 51 건


In [6]:
# 첫인상 — 어제 했던 진단을 한 번 더, 빠르게
print("=== customers ===")
display(customers.head())
print("\n=== orders ===")
display(orders.head())

=== customers ===


,customer_id,age,gender,region,membership,income
0,C0001,39,M,경기,premium,6000.0
1,C0002,34,F,부산,basic,4500.0
2,C0003,41,F,서울,premium,8500.0
3,C0004,49,M,대구,vip,3500.0
4,C0005,33,M,서울,basic,2500.0



=== orders ===


,order_id,customer_id,product_id,category,quantity,amount,channel
0,O00001,C0111,P004,도서,1,29900.0,app
1,O00002,C0014,P029,패션,3,149700.0,app
2,O00003,C0231,P009,식품,2,39800.0,app
3,O00004,C0084,P027,도서,1,29900.0,web
4,O00005,C0130,P026,식품,2,19800.0,app


In [7]:
# 예제: .loc — 라벨(이름)로 가리키기
# customers의 인덱스는 0,1,2,... 정수이지만 '라벨'로 취급됩니다.

# 5번 라벨 행, 'age' 열 한 칸
print("customers.loc[5, 'age']  =", customers.loc[5, "age"])

# 0~3번 라벨 행, customer_id와 age 두 열만 (끝 포함!)
display(customers.loc[0:3, ["customer_id", "age"]])

customers.loc[5, 'age']  = 999


,customer_id,age
0,C0001,39
1,C0002,34
2,C0003,41
3,C0004,49


In [8]:
# 예제: 불리언 인덱싱 — 조건에 맞는 행 전부
# 조건(불리언 Series) 만들기
is_seoul = customers["region"] == "서울"
print("타입:", type(is_seoul).__name__, "/ True 개수:", is_seoul.sum())
print("앞 5개 미리보기:", is_seoul.head().tolist())

# 그 조건을 [ ]에 그대로 — 어제 본 패턴
seoul_customers = customers[is_seoul]
print("\n서울 고객 수:", len(seoul_customers))
display(seoul_customers.head(3))

타입: Series / True 개수: 62
앞 5개 미리보기: [False, False, True, False, True]

서울 고객 수: 62


,customer_id,age,gender,region,membership,income
2,C0003,41,F,서울,premium,8500.0
4,C0005,33,M,서울,basic,2500.0
5,C0006,999,F,서울,basic,NaN


In [9]:
# 예제: .loc + 불리언 — '조건 만족 행의 특정 열만' 가리키기 (오늘의 핵심 패턴)
# 30대 서울 고객의 멤버십만 꺼내기
mask = (customers["region"] == "서울") & (customers["age"].between(30, 39))
display(customers.loc[mask, ["customer_id", "age", "membership"]].head())

,customer_id,age,membership
4,C0005,33,basic
11,C0012,31,basic
21,C0022,33,basic
25,C0026,36,premium
41,C0042,37,basic


In [24]:
# 스스로 해보자! (1)
# 1) 15번 라벨 행, age 열
print(customers.loc[15, 'age'])

# 2) 패션 카테고리 + 수량 2 이상 주문의 order_id, amount 앞 5행
mask = (orders["category"] == '패션') & (orders["quantity"] >= 2)
display(orders.loc[mask, ['order_id', 'amount']].head())

# .loc[0:5] → 6개 (끝 포함), .iloc[0:5] → 5개 (끝 미포함)
# 라벨은 사람이 부르는 이름이라 자연스러운 '끝 포함'을, 위치는 파이썬 슬라이스 관례인 '끝 미포함'을 따릅니다.
display(orders.loc[0:5, ['order_id', 'amount']])
display(orders.iloc[0:5, [0,5]])

120


,order_id,amount
1,O00002,149700.0
74,O00075,149700.0
85,O00086,19800.0
97,O00098,19800.0
108,O00109,19800.0


,order_id,amount
0,O00001,29900.0
1,O00002,149700.0
2,O00003,39800.0
3,O00004,29900.0
4,O00005,19800.0
5,O00006,129900.0


,order_id,amount
0,O00001,29900.0
1,O00002,149700.0
2,O00003,39800.0
3,O00004,29900.0
4,O00005,19800.0


In [25]:
# 예제: query — 문자열로 표현하는 필터
# 30대이면서 서울에 사는 프리미엄/VIP 고객
result = customers.query("age.between(30, 39) and region == '서울' and membership in ['premium', 'vip']")
print("해당 고객 수:", len(result))
display(result.head(3))

해당 고객 수: 4


,customer_id,age,gender,region,membership,income
25,C0026,36,M,서울,premium,2500.0
88,C0089,30,F,서울,premium,NaN
137,C0138,32,F,서울,premium,NaN


In [26]:
# NaN과 비교는 항상 False 입니다 — 직관과 다름
sample = orders[["order_id", "amount"]].head(10).copy()
sample.loc[2, "amount"] = np.nan
display(sample)

print("\n== amount > 0 인 행 ==")
display(sample[sample["amount"] > 0])     # NaN 행은 제외됨 (조건이 False로 평가)

print("\n== amount <= 0 인 행 ==")
display(sample[sample["amount"] <= 0])    # 역시 NaN은 제외됨

,order_id,amount
0,O00001,29900.0
1,O00002,149700.0
2,O00003,NaN
3,O00004,29900.0
4,O00005,19800.0
5,O00006,129900.0
6,O00007,39800.0
7,O00008,29900.0
8,O00009,89900.0
9,O00010,9900.0



== amount > 0 인 행 ==


,order_id,amount
0,O00001,29900.0
1,O00002,149700.0
3,O00004,29900.0
4,O00005,19800.0
5,O00006,129900.0
6,O00007,39800.0
7,O00008,29900.0
8,O00009,89900.0
9,O00010,9900.0



== amount <= 0 인 행 ==


,order_id,amount


In [30]:
# 해결책 1) 결측을 명시적으로 포함하기
mask_with_na = (sample["amount"] > 0) | (sample["amount"].isnull())
print("결측 포함 필터:")
display(sample[mask_with_na])

# 해결책 2) 결측을 명시적으로 제외하기
mask_no_na = (sample["amount"] > 0) & (sample["amount"].notna())
print("\n결측 명시적 제외:")
display(sample[mask_no_na])
display(sample[mask_no_na].dropna()) #dropna 연습

결측 포함 필터:


,order_id,amount
0,O00001,29900.0
1,O00002,149700.0
2,O00003,NaN
3,O00004,29900.0
4,O00005,19800.0
5,O00006,129900.0
6,O00007,39800.0
7,O00008,29900.0
8,O00009,89900.0
9,O00010,9900.0



결측 명시적 제외:


,order_id,amount
0,O00001,29900.0
1,O00002,149700.0
3,O00004,29900.0
4,O00005,19800.0
5,O00006,129900.0
6,O00007,39800.0
7,O00008,29900.0
8,O00009,89900.0
9,O00010,9900.0


,order_id,amount
0,O00001,29900.0
1,O00002,149700.0
3,O00004,29900.0
4,O00005,19800.0
5,O00006,129900.0
6,O00007,39800.0
7,O00008,29900.0
8,O00009,89900.0
9,O00010,9900.0


In [31]:
# 스스로 해보자! (2)
# 1) isin + notna 조합
mask = orders["category"].isin(['패션', '뷰티'])
print("패션,뷰티 전체 건수:", mask.sum())
mask = orders["category"].isin(['패션', '뷰티']) & orders["amount"].notna()
print("결측지 제외한 건수:", mask.sum())
mask = orders["category"].isin(['패션', '뷰티']) & orders["amount"]<0
print("amount 0 이하 건수:", mask.sum())
mask = orders["category"].isin(['패션', '뷰티']) & orders["amount"].isnull()
print("결측치 건수:", mask.sum())

# 2) query로 같은 조건
print('조건 일치 행열 개수',orders.query("category in ['패션', '뷰티'] and amount.notna()").shape)

패션,뷰티 전체 건수: 487
결측지 제외한 건수: 476
amount 0 이하 건수: 0
결측치 건수: 11
조건 일치 행열 개수 (476, 7)


In [32]:
# 5종 세트 ② — 열별 결측 비율(%) — '많고 적음'을 직관적으로
def missing_summary(df):
    s = df.isnull().sum()
    p = (df.isnull().mean() * 100).round(2)
    out = pd.DataFrame({"missing": s, "missing_pct(%)": p})
    return out[out["missing"] > 0].sort_values("missing", ascending=False)

print("[customers]")
display(missing_summary(customers))
print("[orders]")
display(missing_summary(orders))

[customers]


,missing,missing_pct(%)
income,46,15.33
gender,5,1.67


[orders]


,missing,missing_pct(%)
amount,51,2.55


In [36]:
# 5종 세트 ③ — 행별 결측 개수: 통째로 비어 있는 '문제 행' 잡기
row_missing = customers.isnull().sum(axis=1)
print("행별 결측 개수 분포:")
print(row_missing.value_counts().sort_index())

print("\n결측이 2개 이상인 행:")
display(customers[row_missing >= 2])

행별 결측 개수 분포:
0    249
1     51
Name: count, dtype: int64

결측이 2개 이상인 행:


,customer_id,age,gender,region,membership,income


In [39]:
def missing_summary(df):
    s = df.isnull().sum()
    p = (df.isnull().mean() * 100).round(2)
    out = pd.DataFrame({"missing": s, "missing_pct(%)": p})
    return out[out["missing"] > 0].sort_values("missing", ascending=False)

# 스스로 해보자! (3)
# 1) orders 결측 요약
display(missing_summary(orders))

# 2) amount 결측 행의 channel 분포 vs 전체 channel 분포
null_rows = orders[orders['amount'].isnull()]
print("결측 행 채널 분포:")
print(null_rows["channel"].value_counts(normalize=True).round(2))
print("\n전체 채널 분포:")
print(orders["channel"].value_counts(normalize=True).round(2))
print('----\napp 에 오류가 있을 가능성')

,missing,missing_pct(%)
amount,51,2.55


결측 행 채널 분포:
channel
app    0.8
web    0.2
Name: proportion, dtype: float64

전체 채널 분포:
channel
app    0.51
web    0.49
Name: proportion, dtype: float64
----
app 에 오류가 있을 가능성


스스로 해보자! ✏️ (4)
정답은 하나가 아닙니다. 일단 실행해보세요.

다음 시나리오를 읽고 어떤 유형(MCAR / MAR / MNAR)인지 추측해보세요.

번호	시나리오	추정 유형
- (a)	설문지 마지막 페이지가 제본 오류로 일부 사람에게만 인쇄가 빠져 마지막 문항이 비었다	? MCAR
- (b)	"당신의 만성 질환이 있다면 적어주세요"에 질환이 없는 사람들이 비웠다	? MNAR
- (c)	모바일 앱 사용자 그룹에서 특정 화면 버그로 인해 한 항목이 더 자주 비었다	? MAR
- 위 셋 중 단순 평균 대체가 가장 위험한 경우는 어디인가요? b

✅ 짚고 넘어가기

▸ MCAR · MAR · MNAR을 한 문장씩으로 구분해 설명할 수 있나요?
- MCAR: 다른 컬럼에 의한 상관 관계 없음
- MAR: 다른 컬럼에 의한 상관 관계 있음
- MNAR: 컬럼 자체가 결측의 원인. 결측이 다른 요소에 의해 결정된 것이 아닌 자체가 어떤 현상을 암시할 때

▸ 데이터로 검사 가능한 신호와 도메인 지식 중, MNAR은 어느 쪽에 더 의존하나요? 왜 그럴까요?
- 도메인 지식에 의존. 데이터의 인과 관계가 아닌 해당 분야의 지식을 통해 숙고하고 가설을 세워야할 가능성 존재

▸ MNAR로 의심될 때 피해야 하는 처리는 무엇인가요?
- 데이터 맥락 이해와 도메인 지식 없이 섣부른 일반화의 오류를 통해 결측치를 메우는 행동이나 태도, 처리 방

In [40]:
# 처리 결과를 통합한 '정제된 customers' 만들기 — 결정과 근거를 코드로 옮기기
customers_clean = customers.copy()

# 1) age 이상치(0 미만 또는 120 이상)를 NaN으로 표시 — 다음 Part(이상치)에서 더 다룸
customers_clean.loc[
    (customers_clean["age"] < 0) | (customers_clean["age"] >= 120),
    "age"
] = np.nan

# 2) age 결측 → 중앙값 대체 (분포가 좌우 대칭에 가까우므로 평균도 가능)
age_median = customers_clean["age"].median()
customers_clean["age"] = customers_clean["age"].fillna(age_median)

# 3) gender 결측 → 'unknown' 범주로 보존 (MCAR로 추정되나 5건 적어 정보 손실 회피)
customers_clean["gender"] = customers_clean["gender"].fillna("unknown")

# 4) income 결측 → 결측 플래그 + 중앙값 대체 (MNAR 의심을 모델/분석이 인지하도록)
customers_clean["income_missing"] = customers_clean["income"].isnull().astype(int) # isnull 은 boolean 값이 므로 N/A 면 참인 1 값 반환
customers_clean["income"] = customers_clean["income"].fillna(customers_clean["income"].median())

print("처리 전 결측:")
print(customers.isnull().sum())
print("\n처리 후 결측:")
print(customers_clean.isnull().sum())
print("\nincome_missing 컬럼이 추가되었습니다:")
display(customers_clean[["customer_id", "age", "gender", "income", "income_missing"]].head(8))

처리 전 결측:
customer_id     0
age             0
gender          5
region          0
membership      0
income         46
dtype: int64

처리 후 결측:
customer_id       0
age               0
gender            0
region            0
membership        0
income            0
income_missing    0
dtype: int64

income_missing 컬럼이 추가되었습니다:


,customer_id,age,gender,income,income_missing
0,C0001,39.0,M,6000.0,0
1,C0002,34.0,F,4500.0,0
2,C0003,41.0,F,8500.0,0
3,C0004,49.0,M,3500.0,0
4,C0005,33.0,M,2500.0,0
5,C0006,36.0,F,4500.0,1
6,C0007,49.0,F,6000.0,0
7,C0008,42.0,M,4500.0,0


In [41]:
# 스스로 해보자! (5)
# 1) 전체 중앙값 대체
orders_a = orders.copy()
orders_a["amount"] = orders_a["amount"].fillna(orders_a["amount"].median())

# 2) 채널별 중앙값 대체
orders_b = orders.copy()
orders_b["amount"] = orders_b["amount"].fillna(
    orders_b.groupby('channel')["amount"].transform('median')
)

# 3) 채널별 평균 비교
print("[원본 — 결측 제외 평균]")
print(orders.groupby("channel")["amount"].mean().round(0))
print("[A — 전체 중앙값 대체 후]")
print(orders_a.groupby("channel")["amount"].mean().round(0))
print("[B — 채널별 중앙값 대체 후]")
print(orders_b.groupby("channel")["amount"].mean().round(0))

[원본 — 결측 제외 평균]
channel
app    88388.0
web    91980.0
Name: amount, dtype: float64
[A — 전체 중앙값 대체 후]
channel
app    87245.0
web    91648.0
Name: amount, dtype: float64
[B — 채널별 중앙값 대체 후]
channel
app    87245.0
web    91648.0
Name: amount, dtype: float64


✅ 짚고 넘어가기

▸ dropna과 fillna은 각각 어떤 상황에 쓰나요?
- dropna: 결측이 적거나 결측을 채울 방법이 없고 없애도 분석에 큰 영향이 없을 때
- fillna: 결측을 채울 적절한 값(평균이나 중앙값)이 있을 때

▸평균 대체와 중앙값 대체 중, 한쪽으로 쏠린 분포에 더 안전한 것은?
- 중앙값

▸ MNAR이 의심될 때 단순 평균 대체 대신 무엇을 고려하나요?
- 도메인 지식, 데이터 맥락 파악, 비지니스 맥락, 가설 세우기, 검증 과정 계획

In [42]:
# IQR 한 줄로 계산해보기 — orders.quantity
q1 = orders["quantity"].quantile(0.25)
q3 = orders["quantity"].quantile(0.75)
iqr = q3 - q1

lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

print(f"Q1 = {q1}")
print(f"Q3 = {q3}")
print(f"IQR = Q3 - Q1 = {iqr}")
print(f"하한(lower fence) = Q1 - 1.5*IQR = {lower}")
print(f"상한(upper fence) = Q3 + 1.5*IQR = {upper}")

# 정의한 경계로 이상치 골라내기
outlier_mask = (orders["quantity"] < lower) | (orders["quantity"] > upper)
outliers = orders[outlier_mask]
print(f"이상치 개수: {len(outliers)}건 (전체의 {len(outliers)/len(orders)*100:.2f}%)")
display(outliers[["order_id", "category", "quantity", "amount"]])

Q1 = 1.0
Q3 = 2.0
IQR = Q3 - Q1 = 1.0
하한(lower fence) = Q1 - 1.5*IQR = -0.5
상한(upper fence) = Q3 + 1.5*IQR = 3.5
이상치 개수: 2건 (전체의 0.10%)


,order_id,category,quantity,amount
7,O00008,뷰티,100,29900.0
1900,O01901,뷰티,50,29900.0


In [43]:
# IQR을 재사용 가능한 함수로 만들기 — 분석가의 일상 도구
def detect_outliers_iqr(series, k=1.5):
    '''IQR 방법으로 이상치 마스크와 경계를 반환합니다.

    Parameters
    ----------
    series : pd.Series  — 수치형 시리즈
    k      : float      — 경계 폭의 IQR 배수 (기본 1.5, 더 엄격하게 보려면 3)

    Returns
    -------
    mask   : pd.Series(bool)  — 이상치 위치 (True)
    bounds : (lower, upper)   — 경계값
    '''
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - k * iqr
    upper = q3 + k * iqr
    mask = (series < lower) | (series > upper)
    return mask, (lower, upper)

# 사용 예 — age에 적용
mask_age, (lo, up) = detect_outliers_iqr(customers["age"])
print(f"age 경계: ({lo:.1f}, {up:.1f}) / 이상치 수: {mask_age.sum()}")
display(customers.loc[mask_age, ["customer_id", "age", "membership"]])

age 경계: (11.0, 59.0) / 이상치 수: 5


,customer_id,age,membership
5,C0006,999,basic
10,C0011,-3,basic
15,C0016,120,basic
209,C0210,70,basic
262,C0263,6,basic


In [49]:
# 스스로 해보자! (6)
# 1) IQR 경계
mask_amt, (lo, up) = detect_outliers_iqr(orders['amount'])
print(f"하한: {lo:.0f} / 상한: {up:.0f}")

# 2) 이상치 개수·비율
n_out = mask_amt.sum()
print(f"이상치: {n_out}건 ({n_out/len(orders)*100:.2f}%)")
display(orders[mask_amt].head())
print('\n이상치 중 큰 값 보기')
display(orders['amount'].nlargest())

하한: -120600 / 상한: 280200
이상치: 46건 (2.30%)


,order_id,customer_id,product_id,category,quantity,amount,channel
71,O00072,C0126,P038,가전,3,389700.0,app
115,O00116,C0019,P030,가전,3,389700.0,web
133,O00134,C0003,P033,뷰티,3,389700.0,web
154,O00155,C0051,P019,도서,3,389700.0,web
195,O00196,C0009,P014,식품,3,389700.0,web



이상치 중 큰 값 보기


,amount
1500,5000000.0
71,389700.0
115,389700.0
133,389700.0
154,389700.0


In [50]:
# 옵션 ② 클리핑(clip) — 분포의 양 끝을 경계값으로 '잘라' 영향 축소
demo_b = orders.copy()
q1 = demo_b["amount"].quantile(0.25)
q3 = demo_b["amount"].quantile(0.75)
iqr = q3 - q1
lo, up = q1 - 1.5*iqr, q3 + 1.5*iqr

# clip — 경계 밖 값을 경계값으로 대체
demo_b["amount_clipped"] = demo_b["amount"].clip(lower=lo, upper=up)

print("처리 전 amount max:", orders["amount"].max())
print("처리 후 amount_clipped max:", demo_b["amount_clipped"].max())

# 큰 5건 비교
display(
    pd.DataFrame({
        "amount": orders["amount"].nlargest(5).values,
        "amount_clipped": demo_b["amount_clipped"].nlargest(5).values,
    })
)

처리 전 amount max: 5000000.0
처리 후 amount_clipped max: 280200.0


,amount,amount_clipped
0,5000000.0,280200.0
1,389700.0,280200.0
2,389700.0,280200.0
3,389700.0,280200.0
4,389700.0,280200.0


In [57]:
def detect_outliers_iqr(series, k=1.5):
    '''IQR 방법으로 이상치 마스크와 경계를 반환합니다.

    Parameters
    ----------
    series : pd.Series  — 수치형 시리즈
    k      : float      — 경계 폭의 IQR 배수 (기본 1.5, 더 엄격하게 보려면 3)

    Returns
    -------
    mask   : pd.Series(bool)  — 이상치 위치 (True)
    bounds : (lower, upper)   — 경계값
    '''
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - k * iqr
    upper = q3 + k * iqr
    mask = (series < lower) | (series > upper)
    return mask, (lower, upper)

# 옵션 ③ 유지 + 플래그 — 가장 안전한 기본값
demo_c = orders.copy()
mask, (lo, up) = detect_outliers_iqr(demo_c["amount"])
demo_c["amount_outlier"] = mask.astype(int)

print("이상치 플래그 분포:")
print(demo_c["amount_outlier"].value_counts())

print("\n[유지된 이상치 행]")
display(demo_c[demo_c["amount_outlier"] == 1].head())

이상치 플래그 분포:
amount_outlier
0    1954
1      46
Name: count, dtype: int64

[유지된 이상치 행]


,order_id,customer_id,product_id,category,quantity,amount,channel,amount_outlier
71,O00072,C0126,P038,가전,3,389700.0,app,1
115,O00116,C0019,P030,가전,3,389700.0,web,1
133,O00134,C0003,P033,뷰티,3,389700.0,web,1
154,O00155,C0051,P019,도서,3,389700.0,web,1
195,O00196,C0009,P014,식품,3,389700.0,web,1


스스로 해보자! ✏️ (7)
정답은 하나가 아닙니다. 일단 실행해보세요.

다음 네 사례에 처리 결정을 정하고 근거를 한 줄씩 적어보세요. 정해진 정답이 아니라 판단과 기록이 핵심입니다.

#	사례	처리(제거/클리핑/유지+플래그)	근거(한 줄)
1.	customers.age = 999 : 제거. 도메인 지식으로 판단 가능한 불가능한 값
2.	customers.age = 120 : (희귀하지만 가능)	유지+플래그. 가능한 값이니 그대로 두어야 함.
3.	orders.amount = 5,000,000 (가전 카테고리)	: 유지+플래그. 가능할 수 있는 값. 향 후 분석
4.	orders.quantity = 100 (일반 소비자)	: 유지+플래그. 일반 소비자 카테고리이지만 기업 데이터가 들어간 입력 오류일 수 있으므로

✅ 짚고 넘어가기

▸ 이상치 처리 3옵션(제거·클리핑·유지+플래그)을 한 줄씩 말할 수 있나요?
제거: 민감하지 않거나 결측 건수가 얼마 안될 때, 가능하지 않은 값으로 명백한 오류라서 채울 값이 없을 때
- 클리핑: 정상 범위를 벗어난 데이터가 있고 분명 잘못된 데이터인데 해당 데이터가 통계에 왜곡을 줄 때
유지+플래그: 향 후 조사를 해봐야 할 때(분석이 난해하거나 가설이 쉽게 세워지지 않을 때). 도메인 지식으로 해결이 안된 오류(입력이나 조사 단계, 자료 수집 앱의 오류 등 향 후 확인해야 하는 사항 등) 등등. 일단 표시하고 유지해야 함.

▸ "결정 못 하겠을 때의 기본값"은 무엇인가요? 왜 그게 가장 안전한가요?
- 유지+플래그. 값을 임의로 대체하거나 없애지 않고 향 후 분석할 수 있으니 안전

▸ 이상치 처리 결과 옆에 세 줄짜리 메모에는 무엇이 들어가나요?
- 처리 결과에 대한 판단 근거를 남겨야 함. 한계 이유, 근거, 향 후 예상 처리 방향 등


In [64]:
# 우리 데이터에 결정을 적용 — 정제된 orders 만들기
orders_clean = orders.copy()

# 1) quantity 이상치 (=100, =50) → 입력 실수로 추정 (일반 소비자가 한 번에 100개 주문은 드묾)
#    → 안전하게 결측 표시 후 중앙값 대체. 진짜 대량 주문이면 customer_id로 별도 확인 가능.
mask_qty, _ = detect_outliers_iqr(orders_clean["quantity"], k=1.5)
orders_clean.loc[mask_qty, "quantity"] = np.nan
orders_clean["quantity"] = orders_clean["quantity"].fillna(orders_clean["quantity"].median()).astype(int)

# 2) amount 이상치 (=5,000,000) → 가전 카테고리의 1건 고가 거래로 보임
#    → 결정 보류: 유지 + 플래그. 평균 계산 시 필요하면 후속에서 분리.
mask_amt, _ = detect_outliers_iqr(orders_clean["amount"])
orders_clean["amount_outlier"] = mask_amt.astype(int)

# 3) amount 결측 → 채널별 중앙값 대체 (MAR 신호 — Part 4·5에서 결정)
orders_clean["amount"] = orders_clean["amount"].fillna(
    orders_clean.groupby("channel")["amount"].transform("median")
)

print("처리 결과:")
print("- quantity 결측:", orders_clean["quantity"].isnull().sum())
print("- amount 결측:", orders_clean["amount"].isnull().sum())
print("- amount_outlier=1 건수:", orders_clean["amount_outlier"].sum())

display(orders_clean.head())
display(orders_clean[mask_qty].head())
display(orders_clean[mask_amt].head())

처리 결과:
- quantity 결측: 0
- amount 결측: 0
- amount_outlier=1 건수: 46


,order_id,customer_id,product_id,category,quantity,amount,channel,amount_outlier
0,O00001,C0111,P004,도서,1,29900.0,app,0
1,O00002,C0014,P029,패션,3,149700.0,app,0
2,O00003,C0231,P009,식품,2,39800.0,app,0
3,O00004,C0084,P027,도서,1,29900.0,web,0
4,O00005,C0130,P026,식품,2,19800.0,app,0


,order_id,customer_id,product_id,category,quantity,amount,channel,amount_outlier
7,O00008,C0028,P017,뷰티,1,29900.0,web,0
1900,O01901,C0118,P017,뷰티,1,29900.0,app,0


,order_id,customer_id,product_id,category,quantity,amount,channel,amount_outlier
71,O00072,C0126,P038,가전,3,389700.0,app,1
115,O00116,C0019,P030,가전,3,389700.0,web,1
133,O00134,C0003,P033,뷰티,3,389700.0,web,1
154,O00155,C0051,P019,도서,3,389700.0,web,1
195,O00196,C0009,P014,식품,3,389700.0,web,1


In [65]:
# 새 데이터셋 — '옷장패션' 주문 (가상)
np.random.seed(11)
n = 1500

partner = pd.DataFrame({
    "order_id": [f"K{str(i).zfill(5)}" for i in range(1, n + 1)],
    "customer_age": np.random.normal(33, 8, n).round().astype(int),
    "category": np.random.choice(["상의", "하의", "신발", "액세서리"], n, p=[0.35, 0.3, 0.2, 0.15]),
    "channel": np.random.choice(["web", "app"], n, p=[0.4, 0.6]),
    "price": np.random.choice([15900, 29900, 49900, 79900, 129900], n),
    "quantity": np.random.choice([1, 1, 1, 2, 2, 3], n),
})
partner["amount"] = partner["price"] * partner["quantity"]
partner["return_amount"] = np.where(
    np.random.rand(n) < 0.07, partner["amount"] * np.random.uniform(0.5, 1.0, n), 0
).round(0)

# 오염 심기
# (a) 나이 이상치 — 입력 실수(0, 999)
partner.loc[partner.sample(3, random_state=1).index, "customer_age"] = 999
partner.loc[partner.sample(2, random_state=2).index, "customer_age"] = 0

# (b) amount 결측 — app 채널에 더 자주 (MAR 시그널)
app = partner["channel"] == "app"
partner.loc[partner[app].sample(frac=0.05, random_state=3).index, "amount"] = np.nan
partner.loc[partner[~app].sample(frac=0.01, random_state=4).index, "amount"] = np.nan

# (c) return_amount 결측은 그대로 (0=환불없음)이라 결측 아님. 단, '관찰 안 됨'을 의도적으로 표현하기 위해
#     price 결측 5건 추가(접속 시점 가격이 누락된 사례)
partner.loc[partner.sample(5, random_state=5).index, "price"] = np.nan

# (d) quantity 이상치(단일 소비자 200개)
partner.loc[partner.sample(1, random_state=6).index, "quantity"] = 200

# (e) amount 극단값(50,000,000짜리 한 건 — '도매 의심')
partner.loc[partner.sample(1, random_state=7).index, "amount"] = 50_000_000

print("옷장패션 데이터 준비 완료:", partner.shape)
partner.head()

옷장패션 데이터 준비 완료: (1500, 8)


,order_id,customer_age,category,channel,price,quantity,amount,return_amount
0,K00001,47,신발,app,29900.0,2,59800.0,45445.0
1,K00002,31,상의,app,129900.0,3,389700.0,0.0
2,K00003,29,상의,web,49900.0,2,99800.0,0.0
3,K00004,12,상의,web,49900.0,3,149700.0,0.0
4,K00005,33,하의,app,129900.0,1,129900.0,0.0


In [66]:
# 시나리오 1 — 진단
print("shape:", partner.shape)
partner.info()
display(partner.describe())

shape: (1500, 8)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1500 entries, 0 to 1499
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   order_id       1500 non-null   object 
 1   customer_age   1500 non-null   int64  
 2   category       1500 non-null   object 
 3   channel        1500 non-null   object 
 4   price          1495 non-null   float64
 5   quantity       1500 non-null   int64  
 6   amount         1449 non-null   float64
 7   return_amount  1500 non-null   float64
dtypes: float64(3), int64(2), object(3)
memory usage: 93.9+ KB


,customer_age,price,quantity,amount,return_amount
count,1500.000000,1495.000000,1500.000000,1.449000e+03,1500.000000
mean,34.903333,60960.869565,1.789333,1.350475e+05,6381.010000
std,43.936525,40275.103681,5.173406,1.313695e+06,28315.037316
min,0.000000,15900.000000,1.000000,1.590000e+04,0.000000
25%,28.000000,29900.000000,1.000000,4.770000e+04,0.000000
50%,33.000000,49900.000000,2.000000,7.990000e+04,0.000000
75%,38.250000,79900.000000,2.000000,1.299000e+05,0.000000
max,999.000000,129900.000000,200.000000,5.000000e+07,322778.000000


In [67]:
# 결측 진단
print("[열별 결측]")
display(missing_summary(partner))

# 결측이 채널과 관련 있는지 (MAR 신호 검사)
amt_null = partner[partner["amount"].isnull()]
print("\n[amount 결측 행의 채널 분포]")
print(amt_null["channel"].value_counts(normalize=True).round(2))
print("\n[전체 채널 분포]")
print(partner["channel"].value_counts(normalize=True).round(2))

[열별 결측]


,missing,missing_pct(%)
amount,51,3.40
price,5,0.33



[amount 결측 행의 채널 분포]
channel
app    0.88
web    0.12
Name: proportion, dtype: float64

[전체 채널 분포]
channel
app    0.61
web    0.39
Name: proportion, dtype: float64


# 시나리오 1 정리

규모: 1500 행, 8열 데이터

결측:
- amount:	51건 (3.40%)
- price:	5건 (0.33%)
- 결측 채널: app 에 88%
- amount 는 app 오류 의심됨. MAR
- price 소수 건. 기타 오류 가능성. MCAR

이상치:
- customer_age     하한=        12.6  상한=        53.6  이상치=18건
- price            하한=    -45100.0  상한=    154900.0  이상치=0건
- quantity         하한=        -0.5  상한=         3.5  이상치=1건
- amount           하한=    -75600.0  상한=    253200.0  이상치=145건
- return_amount    하한=         0.0  상한=         0.0  이상치=122건

# 시나리오 2 — 처리 결정 (Part 4·5·7의 의사결정)
- amount 결측 : 유지+플래그. APP 오류 가능성 높으므로 오류 검토. APP 데이터 검사, 또는 재입력 유도 등을 결정해야 하므로. APP 오류가 아니라면 해결이 어려워짐.
- price 결측	: 카테고리별 중앙값 대체.	비율 0.3%로 적으므로 대체. 카테고리 표본수가 적을 때는 왜곡 발생
- customer_age = 120, 5 : 평균이나 중앙값 대체. 명백한 불가능한 값이라면 입력 오류 추정. 소수라서 민감하지 않을 수 있지만 입력을 다시 받거나 대체하는 수 밖에 없음
- quantity = 200	: 다른 컬럼 확인 후 평균이나 중앙값 대체. 일반 소비자 구매 수를 넘는 수치. 다른 컬럼을 확인하여 진짜 오류인지 판별해야 하므로 섣불리 제거할 수 없다.
- amount = 50,000,000	: 유지+플래그. 이상 수치일 수 있지만 비즈니스적 가능성도 함께 있음. 다른 컬럼과 함께 고객 연락 등의 확인 및 분석 필요



In [68]:
# 시나리오 3 — 처리 코드 (예시 구현)
partner_clean = partner.copy()

# 1) customer_age 물리적 불가능 값 → NaN → 중앙값 대체
unrealistic = (partner_clean["customer_age"] < 1) | (partner_clean["customer_age"] > 110)
partner_clean.loc[unrealistic, "customer_age"] = np.nan
partner_clean["customer_age"] = partner_clean["customer_age"].fillna(
    partner_clean["customer_age"].median()
).astype(int)

# 2) quantity 이상치 → NaN → 중앙값 대체
mask_q, _ = detect_outliers_iqr(partner_clean["quantity"])
partner_clean.loc[mask_q, "quantity"] = np.nan
partner_clean["quantity"] = partner_clean["quantity"].fillna(
    partner_clean["quantity"].median()
).astype(int)

# 3) amount 이상치(50,000,000) → 유지 + 플래그
mask_a, _ = detect_outliers_iqr(partner_clean["amount"])
partner_clean["amount_outlier"] = mask_a.astype(int)

# 4) amount 결측 → 채널별 중앙값 대체 (MAR 가설)
partner_clean["amount"] = partner_clean["amount"].fillna(
    partner_clean.groupby("channel")["amount"].transform("median")
)

# 5) price 결측 → 카테고리별 중앙값 대체
partner_clean["price"] = partner_clean["price"].fillna(
    partner_clean.groupby("category")["price"].transform("median")
)

# 검증 출력
print("[처리 전 후 결측 비교]")
before = partner.isnull().sum()
after = partner_clean[partner.columns].isnull().sum()
display(pd.DataFrame({"before": before, "after": after}))

print("\n[처리 후 customer_age 범위]:",
      partner_clean["customer_age"].min(), "~", partner_clean["customer_age"].max())
print("[amount_outlier=1 건수]:", partner_clean["amount_outlier"].sum())

[처리 전 후 결측 비교]


,before,after
order_id,0,0
customer_age,0,0
category,0,0
channel,0,0
price,5,0
quantity,0,0
amount,51,0
return_amount,0,0



[처리 후 customer_age 범위]: 5 ~ 60
[amount_outlier=1 건수]: 145


# 옷장패션 데이터 정제 보고서

## 1. 데이터 개요
- 행/열: 1,500 행 / 8 열
- 주요 컬럼: order_id, customer_age, category, channel, price, quantity, amount, return_amount

## 2. 진단 결과
- 결측:
amount: 51건 (3.40%)
price: 5건 (0.33%)

- 이상치(IQR):
customer_age 하한= 12.6 상한= 53.6 이상치=18건
price 하한= -45100.0 상한= 154900.0 이상치=0건
quantity 하한= -0.5 상한= 3.5 이상치=1건
amount 하한= -75600.0 상한= 253200.0 이상치=145건
return_amount 하한= 0.0 상한= 0.0 이상치=122건

- 의심되는 결측 유형:
결측 채널: app 에 88%.
amount 는 app 오류 의심됨(MAR).
price는 소수 건. 기타 오류 가능성(MCAR).

## 3. 처리 결정과 근거
- amount 결측 : 평균이나 중앙값 대체. APP 오류 가능성 높음(MAR). APP 데이터 검사 또는 재입력 유도 등 필요. APP 오류가 아니라면 해결이 어려워짐.
- price 결측 : 카테고리별 평균이나 중앙값 대체. 비율 0.3%로 적으므로 대체. 카테고리 표본수가 적을 때는 왜곡 발생
- customer_age = 120, 5 : 평균이나 중앙값 대체. 명백한 불가능한 값이라면 입력 오류 추정. 소수라서 민감하지 않을 수 있지만 입력을 다시 받거나 대체하는 수 밖에 없음
- quantity = 200 : 다른 컬럼 확인 후 평균이나 중앙값 대체. 일반 소비자 구매 수를 넘는 수치. 다른 컬럼을 확인하여 진짜 오류인지 업체인지 판별해야 하므로 섣불리 제거할 수 없다.
- amount = 50,000,000 : 유지+플래그. 이상 수치일 수 있지만 비즈니스적 가능성도 함께 있음. 다른 컬럼과 함께 고객 연락 등의 확인 및 분석 필요

## 4. 처리 후 검증
- 결측 0건(설계상 NaN 유지가 필요한 컬럼 제외)
- customer_age 범위: 12 ~ 53 (정상)
- amount 의 이상치 145건은 유지+플래그 하는게 맞으므로 그대로 보존(향 후 개인이 아닌 기업 고액인지 확인을 위해)

## 5. 후속 권고
- amount 와 quantity 의 이상치 확인을 위해 customer_id 로 과거 거래 이력 확보 후 확인 검증 필수(업체 여부 확인 필요)
- 결측치는 APP 오류 가능성이 매우 높으므로 점검 필요.
- APP 오류 수정 후 후속 조치: 다시 입력 하거나 그것이 불가능하면 적절한 값으로 대체하는 등의 결정 필요
- 이상치가 업체가 아니라면 후속 조치 논의 필요.